# Persona 2 - Recuperacion Clasica (TF-IDF y BM25)
### Proyecto Bimestre 1 - Recuperacion de Informacion

## 1. Importaciones y configuracion

In [1]:
import pandas as pd
import numpy as np
import nltk

In [2]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from collections import defaultdict, Counter

In [3]:
import string
import math

In [4]:
# Descarga de recursos necesarios de NLTK
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to C:\Users\Kevin
[nltk_data]     Alvear\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to C:\Users\Kevin
[nltk_data]     Alvear\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to C:\Users\Kevin
[nltk_data]     Alvear\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

## 2. Carga del corpus
Se utiliza el dataset Reuters-21578 (ModApte split).  
Se eliminan las filas que no tienen texto.

In [5]:
# Cargar el dataset
df = pd.read_csv('../../ModApte_test.csv')

# Eliminar documentos sin texto
df = df.dropna(subset=['text']).reset_index(drop=True)

print(f'Documentos cargados: {len(df)}')
print(f'Columnas disponibles: {df.columns.tolist()}')

Documentos cargados: 3023
Columnas disponibles: ['text', 'text_type', 'topics', 'lewis_split', 'cgis_split', 'old_id', 'new_id', 'places', 'people', 'orgs', 'exchanges', 'date', 'title']


## 3. Preprocesamiento
Se aplica a cada documento:
- Conversion a minusculas
- Tokenizacion
- Eliminacion de signos de puntuacion
- Eliminacion de stopwords en ingles
- Eliminacion de tokens cortos (menor a 3 caracteres)

In [6]:
# Cargar stopwords en ingles
stop_words = set(stopwords.words('english'))

def preprocess(text):
    # Convertir a minusculas
    text = text.lower()

    # Tokenizar el texto
    tokens = word_tokenize(text)

    # Limpiar tokens: eliminar puntuacion, stopwords y tokens cortos
    tokens = [
        t.replace('.', '').replace('-', ' ').strip()
        for t in tokens
        if t not in string.punctuation
        and t not in stop_words
        and len(t) > 2
    ]

    # Eliminar tokens vacios que quedaron tras el reemplazo
    tokens = [t for t in tokens if t]

    return tokens

# Verificar con el primer documento
print('Ejemplo de tokens del primer documento:')
print(preprocess(df['text'].iloc[0])[:20])

Ejemplo de tokens del primer documento:
['mounting', 'trade', 'friction', 'us', 'japan', 'raised', 'fears', 'among', 'many', 'asia', 'exporting', 'nations', 'row', 'could', 'inflict', 'far reaching', 'economic', 'damage', 'businessmen', 'officials']


## 4. Indice invertido y estadisticas
Se construyen estructuras para recuperacion clasica:
- indice invertido (termino -> {doc_id: frecuencia})
- frecuencia de documento (df)
- longitud de documento y promedio

In [7]:
# Preparar columnas de identificación
if 'new_id' in df.columns:
    # Convertir new_id a numérico; los no numéricos se convierten a NaN
    doc_ids = pd.to_numeric(df['new_id'], errors='coerce')
    # Llenar NaN con el índice de la fila (0,1,2,...) convertido a Serie
    df['doc_id'] = doc_ids.fillna(pd.Series(df.index, index=df.index)).astype(int)
else:
    df['doc_id'] = df.index.astype(int)

In [8]:
# Tokenizar todos los documentos (asegúrate de tener la función preprocess definida)
df['tokens'] = df['text'].apply(preprocess)

In [9]:
# Estructuras principales
inverted_index = defaultdict(dict)
doc_term_freqs = {}
doc_len = {}

for _, row in df.iterrows():
    doc_id = int(row['doc_id'])
    tokens = row['tokens']

    term_counts = Counter(tokens)
    doc_term_freqs[doc_id] = term_counts
    doc_len[doc_id] = sum(term_counts.values())

    for term, freq in term_counts.items():
        inverted_index[term][doc_id] = freq

doc_freq = {term: len(postings) for term, postings in inverted_index.items()}
N = len(doc_term_freqs)
avg_doc_len = sum(doc_len.values()) / N

print(f'Documentos indexados: {N}')
print(f'Términos en vocabulario: {len(inverted_index)}')

Documentos indexados: 3023
Términos en vocabulario: 23141


## 5. Modelos clasicos de recuperacion
Se implementan dos modelos:
- TF-IDF + similitud coseno
- BM25

In [13]:
# IDF para TF-IDF
idf_tfidf = {term: math.log((N + 1) / (df_t + 1)) + 1.0 for term, df_t in doc_freq.items()}

# Norma de cada documento (para coseno)
doc_norms = {}
for doc_id, tf in doc_term_freqs.items():
    sq_sum = 0.0
    for term, freq in tf.items():
        w_td = (1.0 + math.log(freq)) * idf_tfidf[term]
        sq_sum += w_td * w_td
    doc_norms[doc_id] = math.sqrt(sq_sum) if sq_sum > 0 else 1.0

def search_tfidf_cosine(query, top_k=10):
    q_tokens = preprocess(query)
    q_tf = Counter(q_tokens)

    q_weights = {}
    for term, freq in q_tf.items():
        if term in idf_tfidf:
            q_weights[term] = (1.0 + math.log(freq)) * idf_tfidf[term]

    if not q_weights:
        return []

    scores = defaultdict(float)
    q_norm_sq = 0.0

    for term, w_tq in q_weights.items():
        q_norm_sq += w_tq * w_tq
        postings = inverted_index.get(term, {})
        idf = idf_tfidf[term]

        for doc_id, freq in postings.items():
            w_td = (1.0 + math.log(freq)) * idf
            scores[doc_id] += w_tq * w_td

    q_norm = math.sqrt(q_norm_sq) if q_norm_sq > 0 else 1.0

    for doc_id in list(scores.keys()):
        scores[doc_id] = scores[doc_id] / (doc_norms[doc_id] * q_norm)

    ranked_docs = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    return ranked_docs[:top_k]

def search_bm25(query, top_k=10, k1=1.5, b=0.75):
    q_tokens = preprocess(query)
    q_tf = Counter(q_tokens)
    scores = defaultdict(float)

    for term, q_freq in q_tf.items():
        postings = inverted_index.get(term)
        if not postings:
            continue

        df_t = doc_freq[term]
        idf = math.log(1.0 + ((N - df_t + 0.5) / (df_t + 0.5)))

        for doc_id, f_td in postings.items():
            dl = doc_len[doc_id]
            denom = f_td + k1 * (1.0 - b + b * (dl / avg_doc_len))
            score = idf * ((f_td * (k1 + 1.0)) / denom)

            # Ajuste simple por frecuencia del termino en consulta
            score *= (q_freq + 1.0) / q_freq
            scores[doc_id] += score

    ranked_docs = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    return ranked_docs[:top_k]

## 6. Consulta de texto libre y ranking
Se ejecuta una consulta y se muestran los resultados ordenados por relevancia.

In [14]:
def show_results(ranked_docs, model_name='Modelo', top_n=5):
    print(f'\n[{model_name}] Top {min(top_n, len(ranked_docs))} resultados')

    if not ranked_docs:
        print('No se encontraron resultados.')
        return

    for rank, (doc_id, score) in enumerate(ranked_docs[:top_n], start=1):
        row = df[df['doc_id'] == doc_id].iloc[0]
        title = str(row['title']) if 'title' in df.columns and pd.notna(row['title']) else ''
        text = str(row['text']).replace('\n', ' ')
        snippet = text[:140] + ('...' if len(text) > 140 else '')

        print(f'{rank}. doc_id={doc_id} | score={score:.5f}')
        if title:
            print(f'   title: {title}')
        print(f'   text : {snippet}')

query = 'trade japan market'

tfidf_results = search_tfidf_cosine(query, top_k=10)
bm25_results = search_bm25(query, top_k=10)

print(f'Consulta: {query}')
show_results(tfidf_results, model_name='TF-IDF + Coseno', top_n=5)
show_results(bm25_results, model_name='BM25', top_n=5)

Consulta: trade japan market

[TF-IDF + Coseno] Top 5 resultados
1. doc_id=568 | score=0.26142
   title: JAPAN BUYS 5,000 TONNES CANADIAN RAPESEED
  Reuterources said.at an undisclosed price for May shipment,
2. doc_id=319 | score=0.24521
   title: JAPAN MINISTRY HAS NO COMMENT ON RICE TALKS REPORT
 on its closed rice market in the...pan had agreed to hold talks
3. doc_id=889 | score=0.23076
   title: TRADE ISSUES STRAINING EC'S PATIENCE WITH JAPAN
 they believe has repeatedly promised major in...th Japan which
4. doc_id=1203 | score=0.21939
   title: U.S. URGES JAPAN TO OPEN FARM MARKET FURTHER
 Washington cut its trade deficit and ease ...er to help
5. doc_id=1222 | score=0.21939
   title: U.S. URGES JAPAN TO OPEN FARM MARKET FURTHER
 Washington cut its trade deficit and ease ...er to help

[BM25] Top 5 resultados
1. doc_id=319 | score=23.29732
   title: JAPAN MINISTRY HAS NO COMMENT ON RICE TALKS REPORT
 on its closed rice market in the...pan had agreed to hold talks
2. doc_id=1203 